# Spilled Energy — TriviaQA on a T4

**Runtime → Change runtime type → T4 GPU**, then `Runtime → Run all`.

The notebook restarts its own kernel once, after the install (cell 5). That is
not optional: `pip install -e .` writes an `__editable__*.pth` into
site-packages, and `.pth` files are only processed at interpreter startup, so
`lm_polygraph` cannot be imported in a kernel that was already running. After
the restart, just **run all again from the top** — every setup cell is
idempotent and the second pass takes seconds.

Each gate fails fast and loudly: a stale module, a silent model swap, a bad
prompt format, or drifting generations all stop the notebook rather than
producing numbers that look fine and mean nothing.

## 1. Confirm you actually got a T4

Deliberately does **not** import torch. The install can change the torch version,
and a module imported now would stay bound in this kernel while every subprocess
got the new one — exactly the split this notebook exists to avoid. torch is
asserted in cell 7, after the restart.

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi',
                      '--query-gpu=name,memory.total,memory.free,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout
print(out)
assert out.strip(), 'nvidia-smi produced nothing: Runtime -> Change runtime type -> T4 GPU'
if 'T4' not in out:
    print('WARNING: expected a T4. Results stay valid; timings will differ.')

## 2. Mount Drive  *(idempotent)*

Every run writes **straight to Drive**. lm-polygraph saves the manager inside a
`finally:` block, so even a run that raises leaves its results behind, and each
run lands before the next starts — a disconnect costs one run, not all of them.

The mount is a VM-level FUSE process, so it survives the kernel restart;
re-running this cell then just reports it is already mounted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/spilled_energy/runs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
DRIVE = DRIVE_OUT.as_posix()   # posix form, for shell interpolation
print('results ->', DRIVE)

## 3. Clone the experiments branch  *(idempotent)*

`spilled-energy-experiments` = the PR branch **plus** `harness/` and this notebook.
The PR is opened from `spilled-energy`, which holds only the StatCalculator, the
Estimator, the tests and the configs.

Guarded on `isdir`, so the second pass is instant. It also pulls, so re-running
the notebook picks up any pushed fixes without a fresh clone.

In [ ]:
REPO_URL = 'https://github.com/neuezeldaa/lm-polygraph'
BRANCH   = 'spilled-energy-experiments'
REPO     = '/content/lm-polygraph'

import os
if not os.path.isdir(REPO):
    !git clone --branch $BRANCH $REPO_URL $REPO
else:
    print('already cloned; pulling latest')
    !git -C $REPO pull --ff-only origin $BRANCH
%cd $REPO
!git log -1 --oneline

## 4. Environment paths  *(idempotent, must run in every kernel)*

`PYTHONPATH` carries the repo root so a `polygraph_eval` subprocess can import
`harness.pooled_baseline` by dotted path. Environment variables do **not** survive
a kernel restart, so this cell has to run again on the second pass — which is why
it is separate from the install.

In [ ]:
import os, sys, sysconfig
os.environ['PYTHONPATH'] = REPO + os.pathsep + os.environ.get('PYTHONPATH', '')
scripts = sysconfig.get_path('scripts')
if scripts not in os.environ['PATH']:
    os.environ['PATH'] = os.environ['PATH'] + os.pathsep + scripts
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('PYTHONPATH =', os.environ['PYTHONPATH'])

## 5. Install  *(idempotent — skips entirely on the second pass)*

Short-circuits when `lm_polygraph` is already importable, so after the restart
this costs nothing.

Note that torch is **not** pinned to an exact version. An earlier revision pinned
`torch==2.6.0`, which forced a multi-GB downgrade of Colab's build and broke
torchvision, the CUDA/driver match, and left the kernel holding a different torch
than its subprocesses. Upstream only requires `>=2.6.0`, which Colab's build
already satisfies.

In [ ]:
import importlib.util

if importlib.util.find_spec('lm_polygraph') is not None:
    print('lm_polygraph already importable -- skipping install')
else:
    print('installing (expect a few minutes on the first pass)')
    !pip install -q -e .
    !pip install -q -r harness/requirements-repro.txt

!which polygraph_eval || echo 'NOTE: not on PATH; run_baselines.py prints a fallback'

## 6. Restart the kernel — READ THIS

Restarts **only if the in-process state is actually stale**, so it cannot loop:
on the second pass everything imports and this cell just prints OK and moves on.

It restarts when either

* `lm_polygraph` is not importable in-process (the `.pth` was written after this
  kernel started), or
* the imported `torch.__version__` differs from the version pip has on disk
  (a stale module object).

### When it restarts, Colab will say the session crashed. That is expected.
### Just run `Runtime → Run all` again. Cells 1–5 will no-op.

In [ ]:
import importlib.util, importlib.metadata as md_, sys

reasons = []
if importlib.util.find_spec('lm_polygraph') is None:
    reasons.append('lm_polygraph not importable in-process '
                   '(editable-install .pth is only read at interpreter startup)')
if 'torch' in sys.modules:
    import torch
    try:
        on_disk = md_.version('torch')
        if torch.__version__.split('+')[0] != on_disk.split('+')[0]:
            reasons.append(f'stale torch: in-process {torch.__version__} '
                           f'vs installed {on_disk}')
    except Exception as e:
        print('could not compare torch versions:', e)

if reasons:
    print('=' * 68)
    print('RESTARTING THE KERNEL because:')
    for r in reasons:
        print('  -', r)
    print()
    print('  >>> Colab will report the session crashed. THAT IS EXPECTED. <<<')
    print('  >>> Then choose  Runtime -> Run all  again.               <<<')
    print('  >>> Cells 1-5 are idempotent and will no-op.              <<<')
    print('=' * 68)
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print('in-process state is consistent with what is installed -- no restart needed')

## 7. Post-restart environment assertions

torch may have changed version during the install, so the CUDA binding is
re-confirmed here rather than inherited from cell 1. Fails loudly if the
in-process torch is stale, CUDA is unavailable, or the device is not a T4.

In [ ]:
import importlib.metadata as md_
import torch

on_disk = md_.version('torch')
print('torch in-process :', torch.__version__)
print('torch on disk    :', on_disk)
assert torch.__version__.split('+')[0] == on_disk.split('+')[0], (
    'STALE TORCH: the kernel holds a different torch than is installed. '
    'Re-run cell 6 to restart.')

from packaging.version import Version
assert Version(torch.__version__.split('+')[0]) >= Version('2.6.0'), (
    f'torch {torch.__version__} is below lm-polygraph\'s required >=2.6.0')

assert torch.cuda.is_available(), 'CUDA not available after restart'
dev = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info()
print('device           :', dev)
print(f'VRAM             : {free/1e9:.2f} GB free / {total/1e9:.2f} GB')
if 'T4' not in dev:
    print(f'WARNING: expected a T4, got {dev!r}. Results stay valid; timings differ.')
assert total / 1e9 > 14, f'unexpectedly small GPU ({total/1e9:.1f} GB)'
print('\nenvironment OK')

## 8. Fail-fast import check  *(now testing the real post-install state)*

Five seconds here beats discovering a broken import inside a subprocess after the
3B model has loaded. Checks both in-process and in a **fresh subprocess** — the
path `polygraph_eval` actually takes, and the one that matters for the ladder's
dotted-path estimators.

In [ ]:
import subprocess, sys, os

from lm_polygraph.estimators import SpilledEnergy
from lm_polygraph.stat_calculators import EnergyCalculator
from harness.pooled_baseline import PooledBaseline
print('in-process OK:', str(SpilledEnergy(variant='spilled', pooling='max')),
      '|', str(PooledBaseline(score='log_likelihood', pooling='max')))

r = subprocess.run([sys.executable, '-c',
                    'from lm_polygraph.utils.factory_estimator import FactoryEstimator;'
                    'f=FactoryEstimator();'
                    'print("subprocess OK:", f("harness.pooled_baseline",'
                    '{"score":"log_likelihood","pooling":"max"}),'
                    'f("SpilledEnergy",{"variant":"spilled","pooling":"max"}))'],
                   capture_output=True, text=True, env=dict(os.environ))
print(r.stdout.strip() or r.stderr.strip()[-2000:])
assert r.returncode == 0, ('subprocess cannot import -- check PYTHONPATH (cell 4) '
                          'before running anything expensive')

## 9. Model provenance

Printed before anything loads the model. `--expect` makes a silent model swap a
hard failure rather than a footnote. Confirm `model.path`, `dtype` and `device`
against the report.

In [ ]:
!python harness/provenance.py --config configs/stage1/eval_triviaqa_qwen.yaml --expect Qwen/Qwen2.5-3B-Instruct

## 10. Unit tests (CPU, seconds)

Cheap proof the install is sane before any long run.

In [ ]:
!python -m pytest test/test_spilled_energy.py -q

## 11. Dev run, n=150 — the gates fire here

This one short run does four jobs:

1. **Accuracy gate** — hard-fails outside 10–90% exact match. Outside that band
   PRR is noise and every downstream number is meaningless. The usual cause is
   prompt format.
2. **Sign check** — a wrong sign shows up as a large *negative* normalized PRR
   (~-0.7), which reads as a broken method rather than an inverted score.
3. **Answer-span validation** (cell 12).
4. **Runtime measurement** (cell 13).

If the accuracy gate fails, **stop and fix the prompt** — do not widen the band.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_qwen.yaml'
       f" --save-dir '{DRIVE}/dev_n150'"
       ' --samples 150 --n-boot 0'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
from harness.gate import gate
gate(cmd)

## 12. Validate the answer-span assumption

The ablation ladder defines the answer window as the whole generation. That is
only defensible if the generation really is a short answer — measured, not
asserted. Reports single-line fraction, length distribution, ceiling-truncation
rate and exact-match rate.

**Re-run this for CoQA.** The assumption may not transfer.

In [ ]:
cmd = ('python harness/validate_answer_span.py'
       f" --save-dir '{DRIVE}/dev_n150'"
       ' --config configs/stage1/eval_triviaqa_qwen.yaml'
       ' --max-new-tokens 20')
from harness.gate import gate
gate(cmd)

## 13. How long will the real runs take?

Projected from the measured n=150 pass, **before** committing to the long runs.
Linear in n, and it double-counts fixed startup, so it slightly overestimates.

In [ ]:
cmd = ('python harness/estimate_runtime.py'
       f" --from '{DRIVE}/dev_n150'"
       ' --label baselines_n1000 --to-n 1000')
from harness.gate import gate
gate(cmd, required=False)
print('\nNOTE: the ladder run has fewer estimators but the same generation cost,')
print('so budget roughly the same again for run B.')

## 14. Terminator A/B, n=150 — run this BEFORE A, B and C

The pooling window currently **includes** the trailing terminator, because
`GreedyProbsCalculator` trims with `length = j + 1`. For TriviaQA that is a
trailing newline **and** the EOS token: on a median 4-token generation, **half**
the pooled values are maximally predictable tokens whose scores are near-constant
across samples and unrelated to correctness.

Both settings run in ONE UEManager, so generations are identical and the window
is the only difference. The pooled baselines get the same treatment — otherwise
the comparison would change two things at once.

~5 minutes, and it decides which configuration the 1.5–2 hour runs should use.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_terminator_ab.yaml'
       f" --save-dir '{DRIVE}/AB_terminator_n150'"
       ' --n-boot 0'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
from harness.gate import gate
gate(cmd, required=False)

## 15. Terminator A/B — paired report

Both columns side by side with the delta, grouped by pooling. `min` and `max`
are expected to move most: `min` can be dominated outright by a terminator, and
`mean` is diluted by it.

**Paste this table back before starting the long runs.** If the effect is
negligible the terminator stays in and gets one sentence in the report; if it is
material the excluded version becomes primary and this becomes an ablation
finding in its own right.

In [ ]:
cmd = ('python harness/terminator_ab_report.py'
       f" --npz '{DRIVE}/AB_terminator_n150/per_sample_seed1.npz'"
       f" --out '{DRIVE}/AB_terminator_n150/terminator_ab.md'")
from harness.gate import gate
gate(cmd, required=False)

## 16. Run A — primary baseline table, n=1000

The mechanically derived `single_pass_cheap` + `single_pass_plus_aux_model` tiers
**and** all Spilled Energy variants, in one `UEManager` — so generations and
generation settings are identical by construction.

If the sign check flagged a variant as inverted, set `cfg: {sign: -1}` in
`configs/stage1/estimators/stage1_baselines.yaml` and commit. Do not patch it here.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_qwen.yaml'
       f" --save-dir '{DRIVE}/A_baselines_n1000'"
       ' --n-boot 1000'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
from harness.gate import gate
gate(cmd)

## 17. Run B — ablation ladder, n=1000

Same window, same three poolings on every rung, so adjacent rungs differ by
exactly one ingredient: pooled log-likelihood → E^l → E^m → ΔE → ΔE_s.

Run A is already saved to Drive before this starts.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_ladder.yaml'
       f" --save-dir '{DRIVE}/B_ladder_n1000'"
       ' --n-boot 1000'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
from harness.gate import gate
gate(cmd)

## 18. Are the two tables comparable? — hard gate

Greedy decoding at a fixed seed *should* make the two runs byte-identical, but
they resolve different stat calculators, and fp16 reductions are not associative.
So it is **verified, not trusted**: sha256 of the generations and the full
quality vector must match exactly.

If this fails, the two tables are about different generations and must not be
placed side by side — the fix is to make the configs agree on
`output_attentions` and re-run, not to proceed.

In [ ]:
cmd = ('python harness/check_run_consistency.py'
       f" --a '{DRIVE}/A_baselines_n1000'"
       f" --b '{DRIVE}/B_ladder_n1000'"
       ' --label-a baselines --label-b ladder')
from harness.gate import gate
gate(cmd)

## 19. Run C — attention baselines, batch_size=1, n=300

`RAUQ` x2, `CSL` and `AttentionScore` need `output_attentions=True`, which
disables transformers' left-padding NaN guard. At `batch_size=1` there is no
padding at all, so no attention row is ever fully masked and the fp16 overflow
cannot occur — the cost of bs=1 is paid by these four methods rather than by
the whole experiment.

This config also uses **eager** attention, overriding the model group's sdpa.
n=300 because bs=1 is roughly 2x slower per sample.

In [ ]:
cmd = ('python harness/run_baselines.py'
       ' --config configs/stage1/eval_triviaqa_attention_bs1.yaml'
       f" --save-dir '{DRIVE}/C_attention_bs1_n300'"
       ' --n-boot 1000'
       ' --expect-model Qwen/Qwen2.5-3B-Instruct')
from harness.gate import gate
gate(cmd)

## 20. Do Runs A and C agree? — SOFT gate, deliberately

**Not** a byte-equality check, unlike cell 16. Run A uses sdpa and Run C uses
eager — unavoidable, since sdpa cannot return attention weights. Those are
different kernels and round differently in fp16, so where the top two candidates
are nearly tied the argmax can legitimately flip. A hash gate would fire on
correct behaviour.

So this reports the mismatch **fraction** and fails only above a tolerance:
a handful of samples is expected kernel noise; several percent means something
is genuinely wrong — check the degeneracy gate passed for *both* runs, since a
collapsed run differs from a healthy one almost everywhere.

**Report the printed fraction alongside the secondary table.**

`--allow-prefix` is sound because `Dataset.subsample` uses `np.random.choice`
under a fixed seed, which is prefix-stable — the n=300 subsample is exactly the
first 300 of the n=1000 one (asserted by `test_subsample_is_prefix_stable`).

In [ ]:
cmd = ('python harness/check_run_consistency.py'
       f" --a '{DRIVE}/A_baselines_n1000'"
       f" --b '{DRIVE}/C_attention_bs1_n300'"
       ' --label-a bs4_sdpa --label-b bs1_eager'
       ' --allow-prefix --max-mismatch-frac 0.02')
from harness.gate import gate
gate(cmd)

## 21. Energy identity gate — `log p = E^m − E^l`

Algebraically exact, but the two sides come from **different forward passes**:
`greedy_log_likelihoods` from incremental decoding with a KV cache, the energies
from a full-sequence teacher-forced prefill. In fp16 those paths diverge, and
the residual is exactly what `dE` inherits.

`dE = Z_{j+1} − theta_j` is a **cancelling difference**: measured on this model,
|theta| ≈ 22.7 and |Z| ≈ 27.1 give |dE| ≈ 4.2, an amplification of ~6.5×. A
0.19 nat logit divergence becomes ~1.2 nats in `dE` before `max` pooling picks
the worst token — which is why two runs differing only in batch size and
attention kernel agreed on `dE` at only rho=0.29.

**If this fails, do not report any `dE`-based result from the run.**

In [ ]:
for tag in ['A_baselines_n1000', 'B_ladder_n1000']:
    cmd = ('python harness/check_energy_identity.py'
           f" --run '{DRIVE}/{tag}'"
           f" --floor-run '{DRIVE}/C_attention_bs1_n300'")
    print('=' * 70); print(tag); print('=' * 70)
    from harness.gate import gate
    gate(cmd)

## 22. Rank-correlation matrix — how many distinct signals are there?

PRR asks which estimator ranks best. This asks the prior question: how many
**distinct measurements** are on the table at all. At n=150 the top of the table
was a single cluster — `SpilledEnergy_marginal_mean` correlated with
`SelfCertainty` at rho=0.982 and with `MeanTokenEntropy` at 0.909, and
`marginal` vs `logit` reached **0.998** under matched pooling. Only 11 distinct
signals among 33 estimators.

That reframes the question from *does SpilledEnergy beat the baselines* to
*is it distinguishable from what the library already shipped*. Run it on the
n=1000 tables and put both the clusters and the CSV in the report.

In [ ]:
for tag in ['A_baselines_n1000', 'B_ladder_n1000']:
    cmd = ('python harness/correlation_matrix.py'
           f" --npz '{DRIVE}/{tag}/per_sample_seed1.npz'"
           f" --out-md '{DRIVE}/{tag}/correlations.md'"
           f" --out-csv '{DRIVE}/{tag}/correlations.csv'")
    print('=' * 70); print(tag); print('=' * 70)
    from harness.gate import gate
    gate(cmd, required=False)

## 23. The reported tables

Primary metric is **normalized PRR@0.5** with bootstrap CIs. Everything on Drive,
so tables can be rebuilt offline on CPU with
`python harness/run_baselines.py --skip-run --save-dir <dir>`.

In [ ]:
from IPython.display import Markdown, display
for tag, label in [('A_baselines_n1000', 'PRIMARY BASELINE TABLE'),
                   ('B_ladder_n1000',   'ABLATION LADDER'),
                   ('C_attention_bs1_n300',
                    'SECONDARY: attention baselines (bs=1, n=300)')]:
    p = DRIVE_OUT / tag / 'prr_0.5_table.md'
    display(Markdown(f'# {label}'))
    display(Markdown(p.read_text() if p.exists() else f'_missing: {p}_'))

In [ ]:
import torch, transformers, datasets, sys
print('python      ', sys.version.split()[0])
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('device      ', torch.cuda.get_device_name(0))
!git rev-parse HEAD